### Local: wan2.1-vace-1.3b

In [ ]:
# $env:HF_HUB_ENABLE_HF_TRANSFER="1"
# python -c "from huggingface_hub import snapshot_download; snapshot_download('Wan-AI/Wan2.1-VACE-1.3B-diffusers', local_dir='D:/huggingface/wan2.1-vace-1.3b', token='...')"

In [ ]:
import os

# Кеш моделей на D: чтобы не забивать C:
os.environ["HF_HOME"] = "D:/huggingface"
os.environ["MODELSCOPE_CACHE"] = "D:/modelscope"

# Проверяем
print("HF_HOME:", os.environ["HF_HOME"])
print("MODELSCOPE_CACHE:", os.environ["MODELSCOPE_CACHE"])

# Проверяем GPU
import torch
print(f"\nCUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

HF_HOME: D:/huggingface
MODELSCOPE_CACHE: D:/modelscope

CUDA: True
GPU: NVIDIA GeForce RTX 4070 Ti SUPER
VRAM: 16.0 GB


In [3]:
import torch
import PIL.Image
from diffusers import AutoencoderKLWan, WanVACEPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video
from PIL import Image

model_id = "D:/huggingface/wan2.1-vace-1.3b"
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
pipe = WanVACEPipeline.from_pretrained(model_id, vae=vae, torch_dtype=torch.bfloat16)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=3.0)
pipe.enable_model_cpu_offload()
print("Модель загружена!")

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/242 [00:00<?, ?it/s]

Модель загружена!


In [6]:
import time

first_frame = Image.open("OUTPUT/img03_clean.png").convert("RGB")

# 480p, но сохраняем пропорции исходника
orig_w, orig_h = first_frame.size
aspect = orig_h / orig_w
# Кратно 16
width = 480
height = round(width * aspect / 16) * 16
print(f"Размер: {width}x{height} (оригинал {orig_w}x{orig_h})")

num_frames = 81  # 4k+1, ~5 сек при 16fps

first_frame_resized = first_frame.resize((width, height))

frames = [first_frame_resized]
frames.extend([PIL.Image.new("RGB", (width, height), (128, 128, 128))] * (num_frames - 1))

mask = [PIL.Image.new("L", (width, height), 0)]
mask.extend([PIL.Image.new("L", (width, height), 255)] * (num_frames - 1))

prompt = "Pikachu dancing happily, waving arms, jumping, spinning around, animated cartoon style, smooth motion, energetic dance moves"
negative_prompt = "static, blurry, distorted, low quality, ugly, deformed, still image"

t0 = time.time()
output = pipe(
    video=frames,
    mask=mask,
    prompt=prompt,
    negative_prompt=negative_prompt,
    height=height,
    width=width,
    num_frames=num_frames,
    num_inference_steps=30,
    guidance_scale=5.0,
    generator=torch.Generator().manual_seed(42),
).frames[0]
elapsed = time.time() - t0

export_to_video(output, "OUTPUT/img03_i2v.mp4", fps=16)
print(f"Видео сохранено: OUTPUT/img03_i2v.mp4")
print(f"Время: {elapsed:.1f}с ({elapsed/60:.1f} мин)")
print(f"Кадров: {num_frames}, длительность: {num_frames/16:.1f}с")


Размер: 480x448 (оригинал 1254x1172)


  0%|          | 0/30 [00:00<?, ?it/s]

Видео сохранено: OUTPUT/img03_i2v.mp4
Время: 704.8с (11.7 мин)
Кадров: 81, длительность: 5.1с


### fal.ai: Kling Video v2.5 Turbo Pro
- ~$0.35 за 5-секундное видео

In [2]:
import os, time
import fal_client
import requests
import imageio
from io import BytesIO
from dotenv import load_dotenv

load_dotenv("../telegram-ai-bots/.env")
os.environ["FAL_KEY"] = os.getenv("FAL_API_KEY", "")

image_url = fal_client.upload_file("OUTPUT/img03_clean.png")

t0 = time.time()
result = fal_client.subscribe("fal-ai/kling-video/v2.5-turbo/pro/image-to-video", arguments={
    "prompt": "character dancing happily, waving arms, jumping, smooth motion, animated cartoon style",
    "image_url": image_url,
    "duration": "5",
    "aspect_ratio": "1:1",
})
elapsed = time.time() - t0

video_url = result["video"]["url"]
video_bytes = requests.get(video_url).content

with open("OUTPUT/img03_i2v_kling.mp4", "wb") as f:
    f.write(video_bytes)

# Конвертируем в GIF для README
reader = imageio.get_reader(BytesIO(video_bytes), format="mp4")
frames = [frame for frame in reader]
imageio.mimsave("OUTPUT/img03_i2v_kling.gif", frames[::3], fps=5, loop=0)
reader.close()

print(f"Время генерации: {elapsed:.1f}с ({elapsed/60:.1f} мин)")
print(f"Видео: {len(video_bytes) / 1024 / 1024:.1f} MB")
print(f"GIF: {os.path.getsize('OUTPUT/img03_i2v_kling.gif') / 1024 / 1024:.1f} MB")
print("Сохранено: OUTPUT/img03_i2v_kling.mp4 + .gif")


Время генерации: 82.3с (1.4 мин)
Видео: 11.3 MB
GIF: 12.0 MB
Сохранено: OUTPUT/img03_i2v_kling.mp4 + .gif
